# Transfer Learning Strategies

**SENTINEL-CXR** — Uncertainty-Aware Chest Radiograph Triage
Deep Learning (MAIB AI 114) · Prof Anshul Gupta · S P Jain School of Global Management, Dubai

| Group member | Student ID |
|---|---|
| Krishna Mathur | AS25DXB018 |
| Atharva Soundankar | AS25DXB020 |
| Yash Petkar | AS25DXB021 |

---

**Syllabus mapping — Week 8: Transfer Learning — Pre-trained Models and Fine-tuning**

Four strategies on identical splits: training from scratch, freezing the backbone
as a feature extractor, full fine-tuning, and progressive unfreezing. Plus a
comparison against DINOv2 self-supervised features.

The interesting question for medical imaging is whether ImageNet features transfer
at all, given radiographs share almost no low-level statistics with natural photos.
Kornblith et al. found transfer benefit correlates with domain similarity, which
predicts a smaller gain here than on a natural-image task.


In [ ]:
# ── Environment ───────────────────────────────────────────────────────
# Runs on Colab free tier (T4). Nothing here needs a paid runtime.
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run(
        [sys.executable, "-m", "pip", "-q", "install",
         "torchxrayvision", "scikit-learn", "seaborn"],
        check=False,
    )

import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt

SEED = 20260812
np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"torch {torch.__version__} | device {DEVICE}")

plt.rcParams.update({
    "figure.dpi": 120, "axes.spines.top": False, "axes.spines.right": False,
    "font.size": 9, "axes.grid": True, "grid.alpha": 0.25,
})
INSTRUMENT, STAT = "#2E9CB8", "#D64541"

In [ ]:
# ── Data ──────────────────────────────────────────────────────────────
# NIH ChestX-ray14: 112,120 frontal radiographs, 30,805 patients, 14 labels.
# Kaggle: https://www.kaggle.com/datasets/nih-chest-xrays/data
#
# In Colab, the fastest route is the Kaggle API:
#   from google.colab import files; files.upload()      # kaggle.json
#   !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
#   !kaggle datasets download -d nih-chest-xrays/data -p /content/nih --unzip

DATA_DIR = os.environ.get("NIH_DIR", "/content/nih")
META = os.path.join(DATA_DIR, "Data_Entry_2017.csv")

PATHOLOGIES = ["Atelectasis","Cardiomegaly","Consolidation","Edema","Effusion",
               "Emphysema","Fibrosis","Hernia","Infiltration","Mass","Nodule",
               "Pleural_Thickening","Pneumonia","Pneumothorax"]

def load_metadata(path=META):
    """Load the label CSV and expand `Finding Labels` into 14 binary columns."""
    df = pd.read_csv(path)
    df.columns = [c.strip() for c in df.columns]
    for p in PATHOLOGIES:
        df[p] = df["Finding Labels"].str.contains(p, regex=False).astype(int)
    df["Patient Age"] = pd.to_numeric(df["Patient Age"], errors="coerce")
    # Ages above ~100 in this dataset are data-entry errors, not centenarians.
    df = df[(df["Patient Age"] > 0) & (df["Patient Age"] < 100)]
    return df

def patient_disjoint_split(df, fracs=(0.70, 0.10, 0.20), seed=SEED):
    """Split by Patient ID — NEVER by image.

    A patient contributes 3-4 follow-up studies. Splitting by image places the
    same patient's scans on both sides of the boundary, so the model can
    memorise the patient rather than the pathology. Every metric then reports a
    number that will not survive contact with a new hospital. This is the most
    common methodological error in published work on ChestX-ray14.
    """
    patients = df["Patient ID"].unique()
    rng = np.random.default_rng(seed)
    rng.shuffle(patients)
    n = len(patients)
    a, b = int(fracs[0] * n), int((fracs[0] + fracs[1]) * n)
    sets = (set(patients[:a]), set(patients[a:b]), set(patients[b:]))
    train, cal, test = (df[df["Patient ID"].isin(s)].copy() for s in sets)
    assert not (set(train["Patient ID"]) & set(test["Patient ID"])), "patient leak"
    return train, cal, test

## 1. The four strategies

In [ ]:
import torchvision.models as tvm

def make_model(strategy, n_classes=14, dropout=0.2):
    pretrained = strategy != "scratch"
    m = tvm.densenet121(weights="IMAGENET1K_V1" if pretrained else None)
    w = m.features.conv0.weight.data.sum(1, keepdim=True)
    m.features.conv0 = nn.Conv2d(1, 64, 7, 2, 3, bias=False)
    if pretrained: m.features.conv0.weight.data = w
    m.classifier = nn.Sequential(nn.Dropout(dropout),
                                 nn.Linear(m.classifier.in_features, n_classes))
    if strategy == "frozen":
        for p in m.features.parameters(): p.requires_grad = False
    return m

def unfreeze_progressively(model, epoch, schedule=(0, 3, 6, 9)):
    """Unfreeze deeper blocks first, shallow last.

    Early layers hold generic edge and texture filters that transfer well;
    later layers hold ImageNet-specific semantics that must be retrained. And
    unfreezing everything at epoch 0 with a high learning rate destroys the
    pretrained features before they can be exploited.
    """
    blocks = ["denseblock4", "denseblock3", "denseblock2", "denseblock1"]
    for i, blk in enumerate(blocks):
        if epoch >= schedule[i]:
            for name, p in model.features.named_parameters():
                if blk in name: p.requires_grad = True

for s in ["scratch", "frozen", "full", "progressive"]:
    m = make_model(s)
    trainable = sum(p.numel() for p in m.parameters() if p.requires_grad)
    total = sum(p.numel() for p in m.parameters())
    print(f"{s:12s} trainable {trainable:>10,} / {total:,} ({trainable/total:.1%})")

## 2. Discriminative learning rates

One learning rate for a pretrained backbone and a randomly initialised head is a mistake — the head needs to move far, the backbone barely at all.

In [ ]:
def param_groups(model, head_lr=1e-3, backbone_lr=1e-4):
    return [
        {"params": model.features.parameters(),   "lr": backbone_lr},
        {"params": model.classifier.parameters(), "lr": head_lr},
    ]

print("Backbone LR is 10x lower than the head's. With a single LR, either the")
print("head learns too slowly or the backbone's pretrained features are washed")
print("out in the first few hundred steps.")

## 3. Data-efficiency curve

The most useful result: how much labelled data each strategy needs.

In [ ]:
def efficiency_protocol(fractions=(0.01, 0.05, 0.10, 0.25, 0.50, 1.00)):
    print("Train each strategy on {1,5,10,25,50,100}% of the training split")
    print("and plot macro AUROC against label count.")
    print()
    print("Expected shape: transfer learning's advantage is LARGEST at small")
    print("fractions and narrows as data grows. That is the practical argument")
    print("for transfer in medical imaging, where labels are the scarce resource,")
    print("not images.")
    return list(fractions)

efficiency_protocol()

---

### References for this notebook

- Kornblith, S., Shlens, J. & Le, Q. V. (2019). Do better ImageNet models transfer better? *CVPR*.
- Oquab, M. et al. (2023). DINOv2: learning robust visual features without supervision. *TMLR*.
- Raghu, M. et al. (2019). Transfusion: understanding transfer learning for medical imaging. *NeurIPS*.

---

*SENTINEL-CXR is a student research prototype. It is not a medical device and
must not be used for clinical decisions.*
